# MTL for DNSSEC: Querying With the SigTag EDNS(0) Option

<div style="border-left: 5px solid #FFC107; background-color: #FFFDF5; padding: 12px; border-radius: 4px; color: #000;">
    <strong>⚠️ Please note:</strong> This notebook builds on <code>1-build_a_mtl.ipynb</code>. Some files used by this notebook are generated by <code>1-build_a_mtl.ipynb</code>.
</div>

This notebook provides a demonstration of the use of the SigTag EDNS(0) option with MTL for DNSSEC, as described in draft-hdong-dnsop-ml-dsa-mtl-dnssec-sigtag-ext-01. 

Demo Scenarios:
>   * <span style="color: #4aa2cf;">**Scenario A: an empty SigTag**</span>
>       * A client queries the A record for a domain with the SigTag EDNS(0) option enabled. Because the client does not have any knowledge of ladders, it includes an empty SigTag EDNS(0) option in its query.
>       * The server determines that the client has provided an empty SigTag EDNS(0) option. Therefore, the server returns a full signature, with an empty SigTag EDNS(0) option to signal its support of SigTag.
>
> ***
> 
>   * <span style="color: #917440;">**Scenario B: a non-matching SigTag**</span>
>       *  A client queries the A record for a domain with the SigTag EDNS(0) option enabled. Because the client has a known ladder that it thinks is associated with this request, it includes the SigTag EDNS(0) option in its query.
>       * The server determines that the client has provided a non-matching SigTag. Therefore, the server returns a full signature, with an empty SigTag EDNS(0) option to signal its support for SigTag.
> 
> ***
>
>   * <span style="color: #776dba;">**Scenario C: a matching SigTag**</span>
>       * A client queries the A record for a domain with the SigTag EDNS(0) option enabled. Because the client has a known ladder that it thinks is associated with this request, it includes the SigTag EDNS(0) option in its query.
>       * The server determines that the client has provided a matching SigTag. Therefore, the server returns a condensed signature, with an empty SigTag EDNS(0) option to signal its support for SigTag.

## 📋 Demo Outline

<div class="alert alert-block alert-info" style="border-left: 20px solid #a8a7a8; border-right: solid #a8a7a8; border-top: solid #a8a7a8; border-bottom: solid #a8a7a8; background-color: #f1f0f2; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #757475;">Background and Demo Configurations</h3>
<ul>
    <li>SigTag</li>
        <ul>
            <li>Definition</li>
            <li>Format</li>
            <li>Deduplication Optimization</li>
        </ul>
    <li>Importing Libraries</li>
    <li>Demo Configuration
        <ul>
            <li>Example DNS A Record</li>
            <li>Hashing Configurations</li>
            <li>Node Set</li>
            <li>Signed Ladder</li>
        </ul>
    </li>
</ul>
</div>

<div class="alert alert-block alert-danger" style="border-left: 20px solid #c7745d; border-right: solid #c7745d; border-top: solid #c7745d; border-bottom: solid #c7745d; background-color: #f7f1ef; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #c97058;">Handling SigTag</h3>
<ul>
    <li><strong>Section 1</strong> Client - Constructing an Empty SigTag EDNS(0) Option</li>
        <ul>
            <li><span style="color: #4aa2cf;"><strong>Scenario A: An Empty SigTag</strong></span></li>
            <li><span style="color: #917440;"><strong>Scenario B: A Non-Matching SigTag</strong></span></li>
            <li><span style="color: #776dba;"><strong>Scenario C: A Matching SigTag</strong></span></li>
        </ul>
    <li><strong>Section 2</strong> Server - Handling of the Empty SigTag EDNS(0) Option</li>
        <ul>
            <li><span style="color: #4aa2cf;"><strong>Scenario A: An Empty SigTag</strong></span></li>
            <li><span style="color: #917440;"><strong>Scenario B: A Non-Matching SigTag</strong></span></li>
            <li><span style="color: #776dba;"><strong>Scenario C: A Matching SigTag</strong></span></li>
        </ul>
</ul>
</div>

<div class="alert alert-block alert-success" style="border-left: 20px solid #5c86a8; border-right: solid #5c86a8; border-top: solid #5c86a8; border-bottom: solid #5c86a8; background-color: #f5f9fc; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #4e7da3;">DNSSEC Signed Data and Signed Ladder</h3>
<ul>
    <li><strong>Section 3</strong> Server - Generating the DNSSEC Signed Data</li>
    <li><strong>Section 4</strong> Server - Obtaining the Signed Ladder</li>
        <ul>
            <li><span style="color: #4aa2cf;"><strong>Scenario A: An Empty SigTag</strong></span></li>
            <li><span style="color: #917440;"><strong>Scenario B: A Non-Matching SigTag</strong></span></li>
            <li><span style="color: #776dba;"><strong>Scenario C: A Matching SigTag</strong></span></li>
        </ul>
</ul>
</div>


<div class="alert alert-block alert-danger" style="border-left: 20px solid #739e51; border-right: solid #739e51; border-top: solid #739e51; border-bottom: solid #739e51; background-color: #f6fcf2; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #5d8043;">Constructing Authentication Path</h3>
<ul>
    <li><strong>Section 5</strong> Server - Constructing the Authentication Path
        <ul>
            <li><strong>Section 5.1</strong> Obtaining the Leaf Node Hash</li>
            <li><strong>Section 5.2</strong> Obtaining the Target Rung</li>
            <li><strong>Section 5.3</strong> Finding the Siblings</li>
            <li><strong>Section 5.4</strong> Building the Authentication Path</li>
        </ul> 
    </li>
</ul>
</div>

<div class="alert alert-block alert-danger" style="border-left: 20px solid #d4ba4a; border-right: solid #d4ba4a; border-top: solid #d4ba4a; border-bottom: solid #d4ba4a; background-color: #fcfbf2; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #c4a519;">Constructing a Response</h3>
<ul>
    <li><strong>Section 6</strong> Server - Constructing a Response</li>
        <ul>
            <li><span style="color: #4aa2cf;"><strong>Scenario A: An Empty SigTag</strong></span></li>
            <li><span style="color: #917440;"><strong>Scenario B: A Non-Matching SigTag</strong></span></li>
            <li><span style="color: #776dba;"><strong>Scenario C: A Matching SigTag</strong></span></li>
        </ul>
</ul>
</div>

<div class="alert alert-block alert-danger" style="border-left: 20px solid #a8a7a8; border-right: solid #a8a7a8; border-top: solid #a8a7a8; border-bottom: solid #a8a7a8; background-color: #f1f0f2; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #757475;">Demo Summary</h3>
<ul>
    <li>End of Demo</li>
</ul>
</div>

## 📖 SigTag
> ### Definition
> A signed ladder is included in every RRSIG in draft-kaizer-dnsop-ml-dsa-mtl-dnssec-03. While this can provide computational and storage efficiency through ladder reuse, it does not reduce network transmission overhead.
> 
> In fact, when a client already knows the ladder required for verification, the server does not need to retransmit it; the server can transmit only the condensed signatures, therefore reducing network transmission overhead. draft-hdong-dnsop-ml-dsa-mtl-dnssec-sigtag-ext defines a SigTag EDNS(0) option that allows a client to signal its capability to use this mechanism or to indicate its knowledge of a specific SigTag.
>
> MTL has the backward compatibility that an authentication path relative to a new ladder can be verified using an old ladder. When a client provides a SigTag value of a signed ladder that is an older version of the ladder the client requests for verification, the server should determine that the ladder the client provides is valid, and thus will respond with only the condensed signature relative to the ladder that the client caches.
>
> ***
> 
> ### Format
> As defined in draft-hdong-dnsop-ml-dsa-mtl-dnssec-sigtag-ext-01, the SigTag value is calculated as as the concatenation of the series identifier (SID) and the right index of the signed ladder:
> ```
> SigTag = (SID | right_index)
> ```
> <br></br>
> This can be represented as follows: 
> ```
> 0                       8                      16
> +-----------------------------------------------+
> |                                               |
> //                     SID                     //
> |                  (32-octets)                  |
> +-----------------------------------------------+
> |                                               |
> |                  Right Index                  |
> |                   (8-octets)                  |
> |                                               |
> +-----------------------------------------------+
> ```
> 
> * SID, a 32-octet series identifier of the node set, as defined in Section 5.1.1 of draft-kaizer-dnsop-ml-dsa-mtl-dnssec-03 
> * Right Index, an 8-octet value denoting the rightmost index of the signed ladder
>  
> ***
> 
> ### Deduplication Optimization
> Another benifit of utilizing the SigTag EDNS(0) option is the deduplication optimization. When a client signals its support for SigTag, the server will perform the deduplication optimization when: 
>    *  Multiple RRSIG records in the response have condensed signatures that can be verified using the same signed ladder, and
>    *  The signed ladder does not match any SigTag advertised by the client.
> 
>
> Specifically, the server will deduplicate its response by including each common signed ladder within a full signature only once. When deduplication is performed, the full signature is present in the first RRSIG. Other RRSIG records within the response will be condensed signatures.

## 📦 Importing Libraries to Run This Demo: 

In [1]:
from IPython.display import display
from rich.console import Console
from rich.panel import Panel
import json
import hashlib
import socket

Helper functions are imported right before execution. 

## ⚙️ Demo Configurations

### (1) Example DNS A Record
This message is a set of bytes that are intended to be signed and verified. 

In [2]:
# RRSIG RDATA 
rrsig_meta = {
    "type_covered": 1,                                 # DNS record type A
    "algorithm": 131,                                  # Assumed algorithm ID
    "labels": 2,                                       # "ns1.example." has two labels
    "original_ttl": 3600,                              # The TTL the signature is based on
    "sig_expiration": 20260203132321,                  # Signature expiration timestamp
    "sig_inception": 20260106132321,                   # Signature inception timestamp
    "key_tag": 46491,                                  # ID of the DNSKEY used for signing
    "signer_name": "example."                          # Signer's name
}

# RRset Data
rrset = [
    {
        "name": "ns1.example.", 
        "type": 1, 
        "class": 1, 
        "rdata": socket.inet_aton("192.168.1.50")
    },
]

### (2) Hashing Configurations
The hash algorithm and security parameters must match the ladder configuration.

<div style="border-left: 5px solid #FFC107; background-color: #FFFDF5; padding: 12px; border-radius: 4px; color: #000;">
    <strong>⚠️ Please note:</strong> Field sizes in following sections depend on the signature algorithms and may vary if other algorithms are used, which requires corresponding modifications to the underlying functions. 
</div>

In [3]:
# Hash algorithm for generating node hashes: this demo supports "shake".
hash_alg = "shake"

# Security parameter n: this demo supports n = 16, 24, or 32.
n = 16

### (3) Node Set
In this demo, we use the node set used in `1-build_a_mtl.ipynb`. This node set contains 10 leaf nodes and is located at `data/nodeset.json`. 

In [4]:
nodeset_filepath = "data/nodeset.json"

with open(nodeset_filepath, "r") as f:
    nodeset_data = json.load(f)

### (4) Signed Ladder
The ladder has not been changed since our last signing and is located at `data/ladder_signed.txt`. 

In [5]:
with open("data/ladder_signed.txt", "r") as f:
    ladder_readin = f.read()
signed_ladder = bytes.fromhex(ladder_readin)

## 1. Client - Constructing the SigTag EDNS(0) Option

The client queries for the A record of the domain with the SigTag EDNS(0) option. 

<div class="alert alert-block alert-success" style="border-left: 20px solid #4aa2cf; border-right: solid #4aa2cf; border-top: solid #4aa2cf; border-bottom: solid #4aa2cf; background-color: #f4f8fa; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #477e9c;">Scenario A: An Empty SigTag</h3>

Because the client does not have any known ladders, it includes an **empty** SigTag EDNS(0) option in its DNS query, as shown below:
<pre style="background-color: #f4f8fa; border: 1px solid #4aa2cf; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
   0                       8                      16
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |               OPTION-CODE = TBD               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |             OPTION-LENGTH = 0x0000            |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
</pre>
</div>

<div class="alert alert-block alert-success" style="border-left: 20px solid #8a6d3b; border-right: solid #8a6d3b; border-top: solid #8a6d3b; border-bottom: solid #8a6d3b; background-color: #fcf8e3; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #66512c;">Scenario B: A Non-Matching SigTag</h3>

Because the client has a known ladder that it assumes is valid for this request (which it is not), it first calcuates the SigTag value of this ladder (cached on the client side) as: 
<pre style="background-color: #fcf8e3; border: 1px solid #8a6d3b; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
SigTag = (SID | right_index)
</pre>
</div>

In [6]:
from helper.mtl_parser import Ladder

with open("data/ladder_nonmatching.txt", "r") as f:
    ladder_readin_B = f.read()
ladder_readin_bytes_B = bytes.fromhex(ladder_readin_B)
ladder_cached_B = Ladder(fullsig=ladder_readin_bytes_B, n=n)

In [7]:
from helper.utils import print_data

sigtag_client_B = ladder_cached_B.sid + ladder_cached_B.rindex
print_data("Scenario B: SigTag Value of Client Cached Ladder", 
           sigtag_client_B.hex(), len(sigtag_client_B), n=n, print_type="sigtag")

┌─────────────────────────────────────┐
│  ■ SID   ■ Ladder Right Index       │
└─────────────────────────────────────┘

╭──────────── Scenario B: SigTag Value of Client Cached Ladder ─────────────╮
│                                                                           │
│  CD554C26 3F7DEC96 E67106A7 1CD54D41 78600DAA DA33899C E1774298 D19070AB  │
│  00000000 00000007                                                        │
│                                                                           │
│                                                                           │
│  Length of Scenario B: SigTag Value of Client Cached Ladder: 40 bytes     │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

<div class="alert alert-block alert-success" style="border-left: 20px solid #8a6d3b; border-right: solid #8a6d3b; border-top: solid #8a6d3b; border-bottom: solid #8a6d3b; background-color: #fcf8e3; padding: 15px; margin-bottom: 10px; color: #000;">

The client then includes a SigTag EDNS(0) option in its DNS query, as shown below:
<pre style="background-color: #fcf8e3; border: 1px solid #8a6d3b; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
   0                       8                      16
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |               OPTION-CODE = TBD               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |            OPTION-LENGTH = 0x0028             |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |                                               |
   //         OPTION-DATA = sigtag_client         //
   |                                               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
</pre>
</div>

<div class="alert alert-block alert-warning" style="border-left: 20px solid #776dba; border-right: solid #776dba; border-top: solid #776dba; border-bottom: solid #776dba; background-color: #eeedf7; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #6b63a1;">Scenario C: A Matching SigTag</h3>
    
<pre style="background-color: #eeedf7; border: 1px solid #776dba; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
SigTag = (SID | right_index)
</pre>
</div>

In [8]:
from helper.mtl_parser import Ladder

with open("data/ladder_signed.txt", "r") as f:
    ladder_readin_C = f.read()
ladder_readin_bytes_C = bytes.fromhex(ladder_readin_C)
ladder_cached_C = Ladder(fullsig=ladder_readin_bytes_C, n=n)

In [9]:
from helper.utils import print_data

sigtag_client_C = ladder_cached_C.sid + ladder_cached_C.rindex
print_data("Scenario C: SigTag Value of Client Cached Ladder", 
           sigtag_client_C.hex(), len(sigtag_client_C), n=n, print_type="sigtag")

┌─────────────────────────────────────┐
│  ■ SID   ■ Ladder Right Index       │
└─────────────────────────────────────┘

╭──────────── Scenario C: SigTag Value of Client Cached Ladder ─────────────╮
│                                                                           │
│  8F5CE818 A1DF091D FB8288CC AD6FF450 99A48C46 4F540CAB 05114981 736BB9D0  │
│  00000000 00000009                                                        │
│                                                                           │
│                                                                           │
│  Length of Scenario C: SigTag Value of Client Cached Ladder: 40 bytes     │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

<div class="alert alert-block alert-warning" style="border-left: 20px solid #776dba; border-right: solid #776dba; border-top: solid #776dba; border-bottom: solid #776dba; background-color: #eeedf7; padding: 15px; margin-bottom: 10px; color: #000;">

The client then includes a SigTag EDNS(0) option in its DNS query, as shown below:
<pre style="background-color: #eeedf7; border: 1px solid #776dba; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
   0                       8                      16
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |               OPTION-CODE = TBD               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |            OPTION-LENGTH = 0x0028             |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |                                               |
   //         OPTION-DATA = sigtag_client         //
   |                                               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
</pre>
</div>

## 2. Server - Handling of the SigTag EDNS(0) Option

Upon receiving the client's DNS query, the server determines if a SigTag value is provided and if it is valid for the request.

<div class="alert alert-block alert-success" style="border-left: 20px solid #4aa2cf; border-right: solid #4aa2cf; border-top: solid #4aa2cf; border-bottom: solid #4aa2cf; background-color: #f4f8fa; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #477e9c;">Scenario A: An Empty SigTag</h3>

The server determines that the client has provided an empty SigTag EDNS(0) option and, therefore, prepares a full signature response. 
</div>

<div class="alert alert-block alert-success" style="border-left: 20px solid #8a6d3b; border-right: solid #8a6d3b; border-top: solid #8a6d3b; border-bottom: solid #8a6d3b; background-color: #fcf8e3; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #66512c;">Scenario B: A Non-Matching SigTag</h3>

The server determines that the client has provided a SigTag. The server extracts the SigTag value and compares it against the SigTag value of the ladder that the client requests. 
</div>

In [10]:
with open("data/ladder_signed.txt", "r") as f:
    ladder_servreadin = f.read()
ladder_servreadin_bytes = bytes.fromhex(ladder_servreadin)
ladder_signed = Ladder(fullsig=ladder_servreadin_bytes, n=n)

In [11]:
sigtag_server = ladder_signed.sid + ladder_signed.rindex
print_data("SigTag Value of Server Signed Ladder", sigtag_server.hex(), len(sigtag_server), n=n, print_type="sigtag")

┌─────────────────────────────────────┐
│  ■ SID   ■ Ladder Right Index       │
└─────────────────────────────────────┘

╭────────────────── SigTag Value of Server Signed Ladder ───────────────────╮
│                                                                           │
│  8F5CE818 A1DF091D FB8288CC AD6FF450 99A48C46 4F540CAB 05114981 736BB9D0  │
│  00000000 00000009                                                        │
│                                                                           │
│                                                                           │
│  Length of SigTag Value of Server Signed Ladder: 40 bytes                 │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

To check if the client-provided SigTag is valid for its request, the server checks if the advertised SigTag value matches the SigTag value of the ladder that the client requests. If not, the server needs to check if the **backward compatibility** holds by first determining **if the extracted SID matches the SID of the signed ladder that the client requests**. If so, the server next checks **if the ladder the client caches covers the leaf node that the client requests**. In this demo, the client requests the A record of the domain, which has a leaf index of 9. 

In [12]:
from helper.signer import verify_sigtag
is_sigtag_valid = verify_sigtag(serv_sigtag=sigtag_server, client_sigtag=sigtag_client_B, n=n, target_leaf_index=9)

console = Console()
status_text = f"        SigTag {'[bold green]VALID!![/bold green]' if is_sigtag_valid else '[bold red]INVALID!![/bold red]'}"
console.print(Panel(status_text, title=">>> SigTag Value Checking <<<", border_style="#3b82f6", expand=False))

╭─ >>> SigTag Value Checking <<< ─╮
│         SigTag INVALID!!        │
╰─────────────────────────────────╯

<div class="alert alert-block alert-warning" style="border-left: 20px solid #776dba; border-right: solid #776dba; border-top: solid #776dba; border-bottom: solid #776dba; background-color: #eeedf7; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #6b63a1;">Scenario C: A Matching SigTag</h3>
    
The server determines that the client has provided a SigTag. The server extracts the SigTag value and compares it against the SigTag value of the ladder that the client requests. 
</div>

In [13]:
print_data("SigTag Value of Server Signed Ladder", sigtag_server.hex(), len(sigtag_server), n=n, print_type="sigtag")

┌─────────────────────────────────────┐
│  ■ SID   ■ Ladder Right Index       │
└─────────────────────────────────────┘

╭────────────────── SigTag Value of Server Signed Ladder ───────────────────╮
│                                                                           │
│  8F5CE818 A1DF091D FB8288CC AD6FF450 99A48C46 4F540CAB 05114981 736BB9D0  │
│  00000000 00000009                                                        │
│                                                                           │
│                                                                           │
│  Length of SigTag Value of Server Signed Ladder: 40 bytes                 │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

To check if the client-provided SigTag is valid for its request, the server checks if the advertised SigTag value matches the SigTag value of the ladder that the client requests. If not, the server needs to check if the **backward compatibility** holds by first determining **if the extracted SID matches the SID of the signed ladder that the client requests**. If so, the server next checks **if the ladder the client caches covers the leaf node that the client requests**. In this demo, the client requests the A record of the domain, which has a leaf index of 9. 

In [14]:
from helper.signer import verify_sigtag
is_sigtag_valid = verify_sigtag(serv_sigtag=sigtag_server, client_sigtag=sigtag_client_C, n=n, target_leaf_index=9)

console = Console()
status_text = f"        SigTag {'[bold green]VALID!![/bold green]' if is_sigtag_valid else '[bold red]INVALID!![/bold red]'}"
console.print(Panel(status_text, title=">>> SigTag Value Checking <<<", border_style="#3b82f6", expand=False))

╭─ >>> SigTag Value Checking <<< ─╮
│         SigTag VALID!!          │
╰─────────────────────────────────╯

## 3. Server - Generating the DNSSEC Signed Data
To prepare a response, the server first generates the DNSSEC signed data based on the A record that the client requests. 

In [15]:
from helper.utils import create_dnssec_signed_data

message = create_dnssec_signed_data(rrsig_meta, rrset)
print_data("DNSSEC Signed Data", message.hex(), len(message))

╭─────────────────────────── DNSSEC Signed Data ────────────────────────────╮
│                                                                           │
│  00018302 00000E10 6981F6C9 695D0CC9 B59B0765 78616D70 6C650003 6E733107  │
│  6578616D 706C6500 00010001 00000E10 0004C0A8 0132                        │
│                                                                           │
│                                                                           │
│  Length of DNSSEC Signed Data: 54 bytes                                   │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

## 4. Server - Obtaining the Signed Ladder


<div class="alert alert-block alert-success" style="border-left: 20px solid #4aa2cf; border-right: solid #4aa2cf; border-top: solid #4aa2cf; border-bottom: solid #4aa2cf; background-color: #f4f8fa; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #477e9c;">Scenario A: An Empty SigTag</h3>

The server needs to include the signed ladder in its response to construct a full signature. 
</div>

In [16]:
display(ladder_signed.ladder_print())

Ladder Field,Field Length (bytes),Field Data


<div class="alert alert-block alert-success" style="border-left: 20px solid #8a6d3b; border-right: solid #8a6d3b; border-top: solid #8a6d3b; border-bottom: solid #8a6d3b; background-color: #fcf8e3; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #66512c;">Scenario B: A Non-Matching SigTag</h3>

The server needs to include the signed ladder in its response to construct a full signature. 
</div>

In [17]:
display(ladder_signed.ladder_print(preset="nonmatching"))

Ladder Field,Field Length (bytes),Field Data


<div class="alert alert-block alert-warning" style="border-left: 20px solid #776dba; border-right: solid #776dba; border-top: solid #776dba; border-bottom: solid #776dba; background-color: #eeedf7; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #6b63a1;">Scenario C: A Matching SigTag</h3>
    
The server does **not** need to include the signed ladder in its response; it only needs to prepare a condensed signature as the client already knows the valid ladder.  
</div>

## 5. Server - Constructing the Authentication Path

An authentication path consists of: 

```
                     1 1 1 1 1 1
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5
+-------------------------------+
|             Flags             |
+-------------------------------+
|                               |
//             SID             //
|                               |
+-------------------------------+
|                               |
//          Randomizer         //
|                               |
+-------------------------------+
|                               |
|           Leaf Index          |
|                               |
|                               |
+-------------------------------+
|                               |
|     Target Rung Left Index    |
|                               |
|                               |
+-------------------------------+
|                               |
|    Target Rung Right Index    |
|                               |
|                               |
+-------------------------------+
|       Sibling Hash Count      |
+-------------------------------+
|                               |
//      Sibling Hash Values    //
|                               |
+-------------------------------+
```

In this demo, the requester asks for an A record with a leaf index of 9 in the node set. The randomizer is associated to this specific leaf node and impacts the resulting leaf hash value. 

In [18]:
leaf_index = 9

In [19]:
from helper.ladder_updater import LeafNode

leaf_node = LeafNode(lindex=leaf_index, rindex=leaf_index)
leaf_node.randomizer = bytes.fromhex(nodeset_data["nodes"]["level_0"][leaf_index]["randomizer"])

### 5.1 Obtaining the Leaf Node Hash

In [20]:
leaf_node.node_hash = bytes.fromhex(nodeset_data["nodes"]["level_0"][leaf_node.lindex]["hash"])
display(leaf_node.leaf_print())

"Leaf Node (9, 9)",Value


### 5.2 Obtaining the Target Rung

In [21]:
from helper.ladder_updater import find_target_rung

target_rung = find_target_rung(nodeset=nodeset_data, leaf_index=leaf_node.lindex)

leaf_node.rung_lindex_int = target_rung["left_index"]
leaf_node.rung_rindex_int = target_rung["right_index"]
leaf_node.rung_lindex = leaf_node.rung_lindex_int.to_bytes(8, byteorder="big")
leaf_node.rung_rindex = leaf_node.rung_rindex_int.to_bytes(8, byteorder="big")
leaf_node.rung_hash = bytes.fromhex(target_rung["hash"])
display(leaf_node.targetrung_print())

"Leaf Node (9, 9) Target Rung",Value


### 5.3 Finding the Siblings

In [22]:
from helper.ladder_updater import get_sibling_nodes

sibling_list = get_sibling_nodes(nodeset=nodeset_data, leaf_index=leaf_node.lindex)

leaf_node.siblings = []
leaf_node.siblings_withindex = []

if not sibling_list:
    print(f"\033[1;31mThe Leaf Node ({leaf_node.lindex}, {leaf_node.rindex}) does not have siblings\033[0m.")
else: 
    for idx, sib in enumerate(sibling_list):
        leaf_node.siblings.append(bytes.fromhex(sib["hash"]))
        leaf_node.siblings_withindex.append({
            "left_index": sib["left_index"],
            "right_index": sib["right_index"],
            "hash": bytes.fromhex(sib["hash"])
        })

    display(leaf_node.leafsiblings_print())

"Leaf Node (9, 9) Sibling",Value


### 5.4 Building the Authentication Path

The server next builds the authentication path for this leaf node as follows:

In [23]:
# Flag has to be 0
authpath_bytes = int("0").to_bytes(2, byteorder="big") + \
                 ladder_signed.sid + \
                 leaf_node.randomizer + \
                 leaf_node.lindex.to_bytes(8, byteorder="big") + \
                 leaf_node.rung_lindex + \
                 leaf_node.rung_rindex + \
                 len(leaf_node.siblings).to_bytes(2, byteorder="big")

if leaf_node.siblings:
    for each_sib_hash in leaf_node.siblings:
        authpath_bytes += each_sib_hash

leaf_node.authpath = authpath_bytes
display(leaf_node.authpath_print())

"Leaf Node (9, 9) Authentication Path",Value


## 6. Server - Constructing a Response

<div class="alert alert-block alert-success" style="border-left: 20px solid #4aa2cf; border-right: solid #4aa2cf; border-top: solid #4aa2cf; border-bottom: solid #4aa2cf; background-color: #f4f8fa; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #477e9c;">Scenario A: An Empty SigTag</h3>

The server needs to include a full signature in its response. The full signature should start with the `MTL-Type` of 1, as specified in draft-kaizer-dnsop-ml-dsa-mtl-dnssec-03, followed by components listed below:
<pre style="background-color: #f4f8fa; border: 1px solid #4aa2cf; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
                     1 1 1 1 1 1
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5
+-------------------------------+
|                               |
//     Condensed Signature     //
|     (Authentication Path)     |
+-------------------------------+
|                               |
//        Signed Ladder        //
|                               |
+-------------------------------+
</pre>
</div>

In [24]:
full_sig = int(1).to_bytes(1, byteorder="big") + leaf_node.authpath + signed_ladder
print_data("Full Signature", full_sig.hex(), len(full_sig), n=n, print_type="fullsig")

┌──────────────────────────────────────────────────────────────────────────────────────┐
│  ■ MTL-Type        ■ Flags               ■ SID                  ■ Randomizer         │
│  ■ Leaf Index      ■ Target Rung Index   ■ Sibling Hash Count   ■ Sibling Hash       │
│  ■ Flags           ■ SID                 ■ Rung Count           ■ Rung [1] Index     │
│  ■ Rung [1] Hash   ■ Rung [2] Index      ■ Rung [2] Hash        ■ Signature Length   │
│  ■ Signature                                                                         │
└──────────────────────────────────────────────────────────────────────────────────────┘

╭───────────────────────────── Full Signature ──────────────────────────────╮
│                                                                           │
│  0100008F 5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173  │
│  6BB9D02D 778A35B8 4DE7C873 9B0F3B00 374A3600 00000000 00000900 00000000  │
│  00000800 00000000 00000900 01E90354 F0A10015 D979AE8F 5B3E807D 4700008F  │
│  5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173 6BB9D000  │
│  02000000 00000000 00000000 00000000 078080DF D002E93A C7994F0C B974FFE7  │
│  04000000 00000000 08000000 00000000 09B015B4 429BEC7E 08EE28B2 8A79C453  │
│  A5000009 74AEE67B 06EE4BC5 87BE982E 29AF93DC 156F24E4 936833C1 7703C53F  │
│  8BE57A16 0522CEF5 B9064CBC 1FF46F1E 86A0B08A 3AFC4FC5 4D68A8D5 95ECD4DB  │
│  052B866A 989B422D 4848271A 63CC577B 2D9822FA 0426E738 17E1C43B 7EA7634C  │
│  97C745E4 EB348B3B D859B52D 5D46A213 581F515B FF9AE0EC 9630A52C 6CFE07FB  │
│  C1106573 B6B4B484 5529878A 6EDB5F9E 1E61541A 5A6C8EB3 30072801 15BC512B  │
│  B160E5D8 49C84D39 96228EFA 0200FE71 9B45D0AA CF996B24 D4384BA3 CC4FC290  │
│  27C38902 6170097E F3702B14 55C17AE3 15EC5CB2 20B87AB0 6E98AD98 DC8CFD9D  │
│  BF70BBE7 CAFFF2F0 55DA5E49 511A918F 631CA281 E0ABBEA1 4EFD1141 D74799C5  │
│  7936C3AB 7F86424A E4D2F138 DB338C0D C4D6C6A0 E477D863 220AB801 AC486344  │
│  02292768 DFDE0736 C0B11751 1B42DA18 0B873C59 AB25AC2E F9B6FDCF FE171274  │
│  45B932C1 EFCA7D45 720593C4 3FF75FE2 5A752271 C7AB9D5D FDAE0724 2C485FA5  │
│  6F6E31F9 FB6B6C0E EBDF4A4D 05E9D0E8 05215907 3031AE89 8EEE5BC4 D37EE1BC  │
│  DCFA09E6 7896539A 86D4F47A 3BACEC5D F4635564 233AC2A8 7FD6C8B8 624BE3A2  │
│  368D4E71 76650B9E EEB30282 DFA7AEE3 2F0D57E6 77E7E199 3BE5DD03 C16EA69B  │
│  F41812E5 F9E0124C D3A8FDD7 B33C23B9 F5EB5ACB 8774FEE4 A25B2034 4E08FE04  │
│  [1;38;

<div class="alert alert-block alert-success" style="border-left: 20px solid #4aa2cf; border-right: solid #4aa2cf; border-top: solid #4aa2cf; border-bottom: solid #4aa2cf; background-color: #f4f8fa; padding: 15px; margin-bottom: 10px; color: #000;">

Because the client uses SigTag, the server also needs to include an empty SigTag EDNS(0) option in its response to signal its support for SigTag, as follows:
<pre style="background-color: #f4f8fa; border: 1px solid #4aa2cf; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |               OPTION-CODE = TBD               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |             OPTION-LENGTH = 0x0000            |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
</pre>
<br></br>
The server then responds to the client with the above **2617-byte full signature** and **an empty SigTag EDNS(0) option**. 
</div>

<div class="alert alert-block alert-success" style="border-left: 20px solid #8a6d3b; border-right: solid #8a6d3b; border-top: solid #8a6d3b; border-bottom: solid #8a6d3b; background-color: #fcf8e3; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #66512c;">Scenario B: A Non-Matching SigTag</h3>

The server needs to include a full signature in its response. The full signature should start with the `MTL-Type` of 1, as specified in draft-kaizer-dnsop-ml-dsa-mtl-dnssec-03, followed by components listed below:
<pre style="background-color: #fcf8e3; border: 1px solid #8a6d3b; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
                     1 1 1 1 1 1
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5
+-------------------------------+
|                               |
//     Condensed Signature     //
|     (Authentication Path)     |
+-------------------------------+
|                               |
//        Signed Ladder        //
|                               |
+-------------------------------+
</pre>
</div>

In [25]:
full_sig = int(1).to_bytes(1, byteorder="big") + leaf_node.authpath + signed_ladder
print_data("Full Signature", full_sig.hex(), len(full_sig), n=n, print_type="fullsig")

┌──────────────────────────────────────────────────────────────────────────────────────┐
│  ■ MTL-Type        ■ Flags               ■ SID                  ■ Randomizer         │
│  ■ Leaf Index      ■ Target Rung Index   ■ Sibling Hash Count   ■ Sibling Hash       │
│  ■ Flags           ■ SID                 ■ Rung Count           ■ Rung [1] Index     │
│  ■ Rung [1] Hash   ■ Rung [2] Index      ■ Rung [2] Hash        ■ Signature Length   │
│  ■ Signature                                                                         │
└──────────────────────────────────────────────────────────────────────────────────────┘

╭───────────────────────────── Full Signature ──────────────────────────────╮
│                                                                           │
│  0100008F 5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173  │
│  6BB9D02D 778A35B8 4DE7C873 9B0F3B00 374A3600 00000000 00000900 00000000  │
│  00000800 00000000 00000900 01E90354 F0A10015 D979AE8F 5B3E807D 4700008F  │
│  5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173 6BB9D000  │
│  02000000 00000000 00000000 00000000 078080DF D002E93A C7994F0C B974FFE7  │
│  04000000 00000000 08000000 00000000 09B015B4 429BEC7E 08EE28B2 8A79C453  │
│  A5000009 74AEE67B 06EE4BC5 87BE982E 29AF93DC 156F24E4 936833C1 7703C53F  │
│  8BE57A16 0522CEF5 B9064CBC 1FF46F1E 86A0B08A 3AFC4FC5 4D68A8D5 95ECD4DB  │
│  052B866A 989B422D 4848271A 63CC577B 2D9822FA 0426E738 17E1C43B 7EA7634C  │
│  97C745E4 EB348B3B D859B52D 5D46A213 581F515B FF9AE0EC 9630A52C 6CFE07FB  │
│  C1106573 B6B4B484 5529878A 6EDB5F9E 1E61541A 5A6C8EB3 30072801 15BC512B  │
│  B160E5D8 49C84D39 96228EFA 0200FE71 9B45D0AA CF996B24 D4384BA3 CC4FC290  │
│  27C38902 6170097E F3702B14 55C17AE3 15EC5CB2 20B87AB0 6E98AD98 DC8CFD9D  │
│  BF70BBE7 CAFFF2F0 55DA5E49 511A918F 631CA281 E0ABBEA1 4EFD1141 D74799C5  │
│  7936C3AB 7F86424A E4D2F138 DB338C0D C4D6C6A0 E477D863 220AB801 AC486344  │
│  02292768 DFDE0736 C0B11751 1B42DA18 0B873C59 AB25AC2E F9B6FDCF FE171274  │
│  45B932C1 EFCA7D45 720593C4 3FF75FE2 5A752271 C7AB9D5D FDAE0724 2C485FA5  │
│  6F6E31F9 FB6B6C0E EBDF4A4D 05E9D0E8 05215907 3031AE89 8EEE5BC4 D37EE1BC  │
│  DCFA09E6 7896539A 86D4F47A 3BACEC5D F4635564 233AC2A8 7FD6C8B8 624BE3A2  │
│  368D4E71 76650B9E EEB30282 DFA7AEE3 2F0D57E6 77E7E199 3BE5DD03 C16EA69B  │
│  F41812E5 F9E0124C D3A8FDD7 B33C23B9 F5EB5ACB 8774FEE4 A25B2034 4E08FE04  │
│  [1;38;

<div class="alert alert-block alert-success" style="border-left: 20px solid #8a6d3b; border-right: solid #8a6d3b; border-top: solid #8a6d3b; border-bottom: solid #8a6d3b; background-color: #fcf8e3; padding: 15px; margin-bottom: 10px; color: #000;">

Because the client uses SigTag, the server also needs to include an empty SigTag EDNS(0) option in its response to signal its support for SigTag, as follows:
<pre style="background-color: #fcf8e3; border: 1px solid #8a6d3b; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |               OPTION-CODE = TBD               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |             OPTION-LENGTH = 0x0000            |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
</pre>
<br></br>
The server then responds to the client with the above **2617-byte full signature** and **an empty SigTag EDNS(0) option**. 
</div>

<div class="alert alert-block alert-warning" style="border-left: 20px solid #776dba; border-right: solid #776dba; border-top: solid #776dba; border-bottom: solid #776dba; background-color: #eeedf7; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #6b63a1;">Scenario C: A Matching SigTag</h3>
    
Because the server does not need to include the signed ladder in the RRSIG, the RRSIG **contains only the authentication path**. The condensed signature should start with the `MTL-Type` of 2, as specified in draft-hdong-dnsop-ml-dsa-mtl-dnssec-sigtag-ext-01. 

</div>

In [26]:
cond_sig = int(2).to_bytes(1, byteorder="big") + leaf_node.authpath
print_data("Condensed Signature", cond_sig.hex(), len(cond_sig), n=n, print_type="condsig")

┌───────────────────────────────────────────────────────────────────────────────┐
│  ■ MTL-Type     ■ Flags               ■ SID                  ■ Randomizer     │
│  ■ Leaf Index   ■ Target Rung Index   ■ Sibling Hash Count   ■ Sibling Hash   │
└───────────────────────────────────────────────────────────────────────────────┘

╭─────────────────────────── Condensed Signature ───────────────────────────╮
│                                                                           │
│  0200008F 5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173  │
│  6BB9D02D 778A35B8 4DE7C873 9B0F3B00 374A3600 00000000 00000900 00000000  │
│  00000800 00000000 00000900 01E90354 F0A10015 D979AE8F 5B3E807D 47        │
│                                                                           │
│                                                                           │
│  Length of Condensed Signature: 93 bytes                                  │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

<div class="alert alert-block alert-warning" style="border-left: 20px solid #776dba; border-right: solid #776dba; border-top: solid #776dba; border-bottom: solid #776dba; background-color: #eeedf7; padding: 15px; margin-bottom: 10px; color: #000;">
    
The server also needs to include an empty SigTag EDNS(0) option in its response to signal its support for SigTag, as follows:
<pre style="background-color: #eeedf7; border: 1px solid #776dba; padding: 12px; border-radius: 4px; font-family: monospace; font-size: 13px; color: #111; line-height: 1.3;">
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |               OPTION-CODE = TBD               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |             OPTION-LENGTH = 0x0000            |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
</pre>
<br></br>
The server then responds to the client with the **93-byte condensed signature** and **an empty SigTag EDNS(0) option**. 
</div>

## 📝 End of Demo

This demo shows how a client constructs, and how a server handles, an empty SigTag EDNS(0) option. 